# 📊 Lab 3A — Analyze a Query Log Dataset
> **Block 3 | 20 Minutes | Data Source: CSV Query Log**

---

## What This Lab Is About

In Lab 2A you built a working RAG pipeline — documents were indexed, queries were answered, and you observed how retrieval scores vary with query relevance. But a single test query tells you very little about how the system performs across the full range of questions real users ask.

This lab shifts perspective from **building** to **analysing**. You have a log of 500 real queries submitted to the RAG system, each with its retrieval score. Your job is to find the patterns — which types of questions consistently fail retrieval, and why. You will use sentence embeddings, K-means clustering, and UMAP dimensionality reduction to turn 500 rows of text into an actionable map of content gaps.

The output of this lab is not a model or a pipeline — it is a **diagnosis**: three specific content gap clusters with written explanations and concrete fixes.

---

## 🎯 Learning Objectives

- Load and explore a real enterprise RAG query log with pandas
- Embed query text into dense vectors using `sentence-transformers`
- Apply K-means clustering to group semantically similar queries
- Visualise clusters in 2D using UMAP dimensionality reduction
- Identify content gap clusters by retrieval score analysis
- Propose concrete pipeline improvements per gap cluster

---

## 🗺️ Lab Structure

| Step | Cell | What You Are Building | Target Time |
|---|---|---|---|
| Config | 1 | Paths, parameters, seed | 1 min |
| Step 1 | 2–3 | Load CSV, explore schema, plot score distribution | 3 min |
| Step 2 | 4 | Embed 500 queries into 384d vectors | 3 min |
| Step 3 | 5–6 | Elbow curve + K-means clustering | 3 min |
| Step 4 | 7–8 | UMAP 2D visualisation (by cluster + by score) | 4 min |
| Step 5 | 9 | Identify top 3 failure clusters | 3 min |
| Step 6 | 10 | Write diagnosis and proposed fixes | 2 min |
| Validate | 11 | Run full validation suite | 1 min |

> ⚠️ Past 20 minutes and stuck? Open `03_solution.ipynb` — all cells are pre-run.

---

## 📁 Data Source: CSV Query Log

| Column | Type | Description |
|---|---|---|
| `query_text` | string | The raw query submitted by the user |
| `timestamp` | datetime | UTC timestamp of the query |
| `user_id` | string | Anonymised user identifier |
| `retrieval_score` | float | Top-1 cosine similarity score from Qdrant (0.0–1.0) |
| `session_id` | string | Session grouping identifier |

---
## ⚙️ Cell 1 — Configuration

All tunable parameters are defined here. The random seed ensures reproducible clustering and UMAP layouts across runs.

In [ ]:
# ============================================================
# CELL 1 — Configuration
# ============================================================
import warnings
warnings.filterwarnings("ignore")

# --- Data source -------------------------------------------------
CSV_PATH = "/data/workshop/query_log.csv"

# --- Embedding model ---------------------------------------------
EMBED_MODEL  = "sentence-transformers/all-MiniLM-L6-v2"
EMBED_DIM    = 384
EMBED_BATCH  = 50

# --- Clustering --------------------------------------------------
N_CLUSTERS   = 8
K_MIN        = 2
K_MAX        = 15
KMEANS_ITER  = 500
KMEANS_INIT  = 10

# --- UMAP --------------------------------------------------------
UMAP_NEIGHBORS = 15
UMAP_MIN_DIST  = 0.1

# --- Analysis thresholds -----------------------------------------
MISS_THRESHOLD     = 0.50   # scores below this are retrieval misses
GAP_CLUSTER_THRESH = 0.55   # clusters with mean score below this are gaps
N_GAP_CLUSTERS     = 3      # number of gap clusters to diagnose
SAMPLE_QUERIES     = 10     # sample queries to print per gap cluster

# --- Reproducibility ---------------------------------------------
RANDOM_SEED = 42

import numpy as np
np.random.seed(RANDOM_SEED)

print("\u2705 Configuration loaded.")
print(f"   CSV path       : {CSV_PATH}")
print(f"   Embed model    : {EMBED_MODEL} ({EMBED_DIM}d)")
print(f"   Embed batch    : {EMBED_BATCH}")
print(f"   K-means k      : {N_CLUSTERS}")
print(f"   UMAP neighbors : {UMAP_NEIGHBORS}, min_dist={UMAP_MIN_DIST}")
print(f"   Miss threshold : {MISS_THRESHOLD}")
print(f"   Random seed    : {RANDOM_SEED}")

---
## 📂 Step 1 — Load the Query Log

### Cell 2 — Load and Inspect the CSV

### Why this step exists

Before any analysis, you need to understand the shape and quality of the data. How many queries are there? Are there missing values? What does the retrieval score distribution look like? A quick exploratory pass catches data quality issues before they silently corrupt downstream analysis.

In [ ]:
# ============================================================
# CELL 2 — Load and Inspect the Query Log CSV
# ============================================================
import pandas as pd
import os

if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(
        f"\u274c CSV not found: {CSV_PATH}\n"
        f"   Ask your instructor to confirm the data path."
    )

df = pd.read_csv(CSV_PATH, sep=",")

# Normalise column names
df.columns = df.columns.str.strip().str.lower()

REQUIRED_COLS = {"query_text", "timestamp", "user_id", "retrieval_score", "session_id"}
missing_cols  = REQUIRED_COLS - set(df.columns)
if missing_cols:
    raise ValueError(
        f"\u274c Missing expected columns: {missing_cols}\n"
        f"   Found columns: {list(df.columns)}"
    )

# Parse timestamp
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")

# Drop rows with null query_text or retrieval_score
before = len(df)
df = df.dropna(subset=["query_text", "retrieval_score"])
df["query_text"] = df["query_text"].astype(str).str.strip()
after = len(df)

print(f"\u2705 CSV loaded successfully.")
print(f"   Rows loaded     : {before}")
print(f"   Rows after clean: {after} ({before - after} dropped)")
print(f"   Columns         : {list(df.columns)}")
print()
print("   First 5 rows:")
display(df.head())
print()
print("   Data types:")
print(df.dtypes)
print()
print("   Retrieval score summary:")
print(df["retrieval_score"].describe().round(4))

---
### Cell 3 — Plot Score Distribution and Calculate Miss Rate

### Why this step exists

The retrieval score distribution is the single most important diagnostic chart for a RAG system. A healthy system shows a distribution skewed toward 1.0. A system with content gaps shows a bimodal distribution — a cluster of high-scoring queries (the corpus covers these topics) and a cluster of low-scoring queries (the corpus does not). The miss rate quantifies the problem.

In [ ]:
# ============================================================
# CELL 3 — Plot Retrieval Score Distribution
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

miss_mask  = df["retrieval_score"] < MISS_THRESHOLD
miss_rate  = miss_mask.mean() * 100
miss_count = miss_mask.sum()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# --- Histogram ---------------------------------------------------
ax = axes[0]
ax.hist(df["retrieval_score"], bins=40,
        color="#1f77b4", edgecolor="white", alpha=0.85)
ax.axvline(MISS_THRESHOLD, color="#d62728", linestyle="--", linewidth=1.5,
           label=f"Miss threshold ({MISS_THRESHOLD})")
ax.set_xlabel("Retrieval Score (cosine similarity)")
ax.set_ylabel("Query Count")
ax.set_title("Retrieval Score Distribution")
ax.legend()
ax.grid(axis="y", alpha=0.3)

# --- CDF ---------------------------------------------------------
ax2 = axes[1]
sorted_scores = df["retrieval_score"].sort_values()
cdf = [i / len(sorted_scores) for i in range(len(sorted_scores))]
ax2.plot(sorted_scores, cdf, color="#1f77b4", linewidth=2)
ax2.axvline(MISS_THRESHOLD, color="#d62728", linestyle="--", linewidth=1.5,
            label=f"Miss threshold ({MISS_THRESHOLD})")
ax2.axhline(miss_rate / 100, color="#ff7f0e", linestyle=":", linewidth=1.5,
            label=f"Miss rate ({miss_rate:.1f}%)")
ax2.set_xlabel("Retrieval Score")
ax2.set_ylabel("Cumulative Fraction")
ax2.set_title("Cumulative Distribution of Retrieval Scores")
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("/tmp/lab3a_score_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\u2705 Score distribution analysis:")
print(f"   Total queries       : {len(df)}")
print(f"   Miss rate (< {MISS_THRESHOLD}) : {miss_count} queries ({miss_rate:.1f}%)")
print(f"   Mean score          : {df['retrieval_score'].mean():.4f}")
print(f"   Median score        : {df['retrieval_score'].median():.4f}")
print(f"   Std deviation       : {df['retrieval_score'].std():.4f}")
print()
print(f"   Interpretation: {miss_rate:.1f}% of queries are retrieval misses.")
if miss_rate > 20:
    print(f"   \u26a0\ufe0f  Miss rate > 20% — significant content gaps detected.")
else:
    print(f"   \u2705 Miss rate within acceptable range.")

---
## 🔢 Step 2 — Embed Queries

### Cell 4 — Encode All 500 Queries into 384d Vectors

### Why this step exists

K-means clustering operates on numerical vectors, not text. To cluster queries by semantic similarity, we first convert each query string into a dense vector using a sentence transformer model. `all-MiniLM-L6-v2` is a compact 22M parameter model that produces 384-dimensional embeddings — small enough to fit in CPU memory for 500 queries while capturing rich semantic meaning.

We batch in groups of 50 to avoid memory pressure. The resulting matrix has shape `(500, 384)` — one row per query, one column per embedding dimension.

In [ ]:
# ============================================================
# CELL 4 — Embed All Queries
# ============================================================
from sentence_transformers import SentenceTransformer
import numpy as np
import time

print(f"Loading embedding model: {EMBED_MODEL}")
embed_model = SentenceTransformer(EMBED_MODEL)
print("\u2705 Model loaded.")
print()

queries    = df["query_text"].tolist()
embeddings = []
t_start    = time.time()

for i in range(0, len(queries), EMBED_BATCH):
    batch = queries[i : i + EMBED_BATCH]
    vecs  = embed_model.encode(
        batch,
        show_progress_bar=False,
        normalize_embeddings=True,
    )
    embeddings.append(vecs)
    done = min(i + EMBED_BATCH, len(queries))
    print(f"   Encoded {done}/{len(queries)} queries...", end="\r")

embeddings = np.vstack(embeddings)
elapsed    = time.time() - t_start

print()
print("\u2705 Embedding complete.")
print(f"   Matrix shape   : {embeddings.shape}")
print(f"   Expected shape : ({len(queries)}, {EMBED_DIM})")
print(f"   Elapsed        : {elapsed:.1f}s")
print(f"   Sample norms   : {np.linalg.norm(embeddings[:3], axis=1).round(4)}")

assert embeddings.shape == (len(queries), EMBED_DIM), (
    f"\u274c Embedding shape mismatch: expected ({len(queries)}, {EMBED_DIM}), "
    f"got {embeddings.shape}"
)
print()
print(f"\u2705 Shape assertion passed: {embeddings.shape}")

---
## 🔵 Step 3 — Cluster with K-means

### Cell 5 — Elbow Curve to Validate k=8

### Why this step exists

The elbow method plots the within-cluster sum of squares (inertia) for a range of k values. As k increases, inertia decreases — but the rate of decrease slows at the point where adding more clusters provides diminishing returns. The 'elbow' in the curve is the optimal k. We expect to see a clear inflection at k=8 for this dataset.

In [ ]:
# ============================================================
# CELL 5 — Elbow Curve
# ============================================================
from sklearn.cluster import KMeans

inertias = []
k_range  = range(K_MIN, K_MAX + 1)

print(f"Computing inertia for k={K_MIN} to k={K_MAX}...")
for k in k_range:
    km = KMeans(
        n_clusters=k,
        max_iter=200,
        n_init=5,
        random_state=RANDOM_SEED,
    )
    km.fit(embeddings)
    inertias.append(km.inertia_)
    print(f"   k={k:2d}  inertia={km.inertia_:.2f}", end="\r")

print()

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(list(k_range), inertias,
        marker="o", color="#1f77b4", linewidth=2, markersize=6)
ax.axvline(N_CLUSTERS, color="#d62728", linestyle="--", linewidth=1.5,
           label=f"Selected k={N_CLUSTERS}")
ax.set_xlabel("Number of Clusters (k)")
ax.set_ylabel("Inertia (Within-Cluster Sum of Squares)")
ax.set_title("K-means Elbow Curve — Validate k=8")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("/tmp/lab3a_elbow.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\u2705 Elbow curve complete.")
print(f"   Inertia at k={N_CLUSTERS}: {inertias[N_CLUSTERS - K_MIN]:.2f}")
print("   Observe the chart: the curve should show a clear inflection near k=8.")

---
### Cell 6 — Fit K-means and Assign Cluster Labels

### Why this step exists

With k=8 validated by the elbow curve, we fit the final K-means model and assign cluster labels back to the dataframe. The cluster size distribution tells us whether the clusters are balanced (good) or heavily skewed (may indicate a dominant topic dominating the corpus).

In [ ]:
# ============================================================
# CELL 6 — Fit K-means (k=8)
# ============================================================
kmeans = KMeans(
    n_clusters=N_CLUSTERS,
    max_iter=KMEANS_ITER,
    n_init=KMEANS_INIT,
    random_state=RANDOM_SEED,
)
kmeans.fit(embeddings)

df["cluster"] = kmeans.labels_

cluster_sizes = df["cluster"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(
    [f"C{i}" for i in cluster_sizes.index],
    cluster_sizes.values,
    color="#1f77b4", edgecolor="white",
)
for bar, val in zip(bars, cluster_sizes.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        str(val), ha="center", va="bottom", fontsize=9
    )
ax.set_xlabel("Cluster")
ax.set_ylabel("Query Count")
ax.set_title(f"K-means Cluster Size Distribution (k={N_CLUSTERS})")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("/tmp/lab3a_cluster_sizes.png", dpi=150, bbox_inches="tight")
plt.show()

print("\u2705 K-means fitted.")
print(f"   Inertia (final)  : {kmeans.inertia_:.2f}")
print(f"   Iterations used  : {kmeans.n_iter_}")
print()
print("   Cluster sizes:")
for cid, cnt in cluster_sizes.items():
    print(f"     Cluster {cid}: {cnt} queries ({cnt / len(df) * 100:.1f}%)")

---
## 🗺️ Step 4 — Visualise with UMAP

### Cell 7 — Reduce to 2D and Plot by Cluster

### Why this step exists

UMAP (Uniform Manifold Approximation and Projection) reduces the 384-dimensional embedding space to 2D while preserving local neighbourhood structure. Queries that are semantically similar end up close together in the 2D plot. Colouring by cluster label shows whether K-means has found geometrically coherent groups — if the cluster colours form distinct blobs, the clustering is meaningful.

In [ ]:
# ============================================================
# CELL 7 — UMAP Dimensionality Reduction + Cluster Plot
# ============================================================
import umap

print("Running UMAP dimensionality reduction (this takes ~30s)...")
reducer = umap.UMAP(
    n_neighbors=UMAP_NEIGHBORS,
    min_dist=UMAP_MIN_DIST,
    n_components=2,
    random_state=RANDOM_SEED,
    metric="cosine",
)
umap_2d = reducer.fit_transform(embeddings)
df["umap_x"] = umap_2d[:, 0]
df["umap_y"] = umap_2d[:, 1]

print(f"\u2705 UMAP complete. Output shape: {umap_2d.shape}")
print()

CLUSTER_COLORS = plt.cm.get_cmap("tab10", N_CLUSTERS)

fig, ax = plt.subplots(figsize=(10, 7))
for cid in range(N_CLUSTERS):
    mask = df["cluster"] == cid
    ax.scatter(
        df.loc[mask, "umap_x"],
        df.loc[mask, "umap_y"],
        c=[CLUSTER_COLORS(cid)],
        label=f"Cluster {cid} (n={mask.sum()})",
        s=18, alpha=0.75,
    )
    cx = df.loc[mask, "umap_x"].mean()
    cy = df.loc[mask, "umap_y"].mean()
    ax.text(
        cx, cy, str(cid),
        fontsize=11, fontweight="bold",
        ha="center", va="center",
        bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.7)
    )

ax.set_title("UMAP Projection — Coloured by K-means Cluster", fontsize=13)
ax.set_xlabel("UMAP Dimension 1")
ax.set_ylabel("UMAP Dimension 2")
ax.legend(loc="upper right", fontsize=8, ncol=2)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig("/tmp/lab3a_umap_clusters.png", dpi=150, bbox_inches="tight")
plt.show()

centroids_2d = np.array([
    [
        df.loc[df["cluster"] == c, "umap_x"].mean(),
        df.loc[df["cluster"] == c, "umap_y"].mean()
    ]
    for c in range(N_CLUSTERS)
])
centroid_spread = np.std(centroids_2d)
print(f"   Centroid spread (std): {centroid_spread:.3f}")
print("   (Higher spread = more visually distinct clusters)")

---
### Cell 8 — UMAP Plot Coloured by Retrieval Score

### Why this step exists

Colouring the same UMAP projection by retrieval score (continuous, red=low, blue=high) reveals which clusters are associated with poor retrieval. If a cluster that appeared as a distinct blob in the previous plot is predominantly red in this plot, that cluster is a content gap — the corpus does not cover the topics those queries ask about.

In [ ]:
# ============================================================
# CELL 8 — UMAP Plot Coloured by Retrieval Score
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Left: continuous score heatmap ------------------------------
ax = axes[0]
sc = ax.scatter(
    df["umap_x"], df["umap_y"],
    c=df["retrieval_score"],
    cmap="RdYlGn",
    vmin=0.0, vmax=1.0,
    s=18, alpha=0.80,
)
plt.colorbar(sc, ax=ax, label="Retrieval Score")
ax.set_title(
    "UMAP — Coloured by Retrieval Score\n(Red = Low Score = Content Gap)",
    fontsize=11
)
ax.set_xlabel("UMAP Dimension 1")
ax.set_ylabel("UMAP Dimension 2")
ax.grid(alpha=0.2)

# --- Right: binary miss/hit --------------------------------------
ax2 = axes[1]
miss_colors = [
    "#d62728" if s < MISS_THRESHOLD else "#1f77b4"
    for s in df["retrieval_score"]
]
ax2.scatter(df["umap_x"], df["umap_y"],
            c=miss_colors, s=18, alpha=0.75)
hit_patch  = mpatches.Patch(color="#1f77b4",
                             label=f"Hit (score \u2265 {MISS_THRESHOLD})")
miss_patch = mpatches.Patch(color="#d62728",
                             label=f"Miss (score < {MISS_THRESHOLD})")
ax2.legend(handles=[hit_patch, miss_patch], loc="upper right")
ax2.set_title(f"UMAP — Hit vs Miss (threshold={MISS_THRESHOLD})", fontsize=11)
ax2.set_xlabel("UMAP Dimension 1")
ax2.set_ylabel("UMAP Dimension 2")
ax2.grid(alpha=0.2)

plt.tight_layout()
plt.savefig("/tmp/lab3a_umap_scores.png", dpi=150, bbox_inches="tight")
plt.show()

print("\u2705 Score overlay plots complete.")
print("   Observation: Identify clusters where red points dominate.")
print("   These are your content gap candidates for Step 5.")

---
## 🔍 Step 5 — Identify Top 3 Failure Clusters

### Cell 9 — Rank Clusters by Mean Retrieval Score

### Why this step exists

Visual inspection of the UMAP plot identifies candidate gap clusters, but we need a quantitative ranking to select the three worst-performing clusters for diagnosis. Sorting clusters by mean retrieval score ascending gives us the bottom 3 — the clusters where the RAG system most consistently fails to retrieve relevant content.

In [ ]:
# ============================================================
# CELL 9 — Identify Top 3 Failure Clusters
# ============================================================

cluster_stats = df.groupby("cluster")["retrieval_score"].agg(
    mean_score="mean",
    median_score="median",
    std_score="std",
    miss_count=lambda x: (x < MISS_THRESHOLD).sum(),
    total_count="count",
).reset_index()
cluster_stats["miss_rate"] = (
    cluster_stats["miss_count"] / cluster_stats["total_count"] * 100
).round(1)
cluster_stats = cluster_stats.sort_values("mean_score")

print("\u2705 Cluster performance ranking (worst first):")
print()
display(cluster_stats.round(4))

# --- Bar chart of mean scores ------------------------------------
fig, ax = plt.subplots(figsize=(10, 4))
bar_colors = [
    "#d62728" if s < GAP_CLUSTER_THRESH else "#1f77b4"
    for s in cluster_stats["mean_score"]
]
bars = ax.bar(
    [f"C{c}" for c in cluster_stats["cluster"]],
    cluster_stats["mean_score"],
    color=bar_colors, edgecolor="white",
)
ax.axhline(GAP_CLUSTER_THRESH, color="#ff7f0e", linestyle="--", linewidth=1.5,
           label=f"Gap threshold ({GAP_CLUSTER_THRESH})")
for bar, val in zip(bars, cluster_stats["mean_score"]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.005,
        f"{val:.3f}", ha="center", va="bottom", fontsize=8
    )
ax.set_xlabel("Cluster (sorted by mean score ascending)")
ax.set_ylabel("Mean Retrieval Score")
ax.set_title("Mean Retrieval Score per Cluster — Gap Clusters in Red")
ax.legend()
ax.set_ylim(0, 1.05)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("/tmp/lab3a_cluster_scores.png", dpi=150, bbox_inches="tight")
plt.show()

# --- Identify gap clusters and print sample queries --------------
gap_clusters = cluster_stats.head(N_GAP_CLUSTERS)["cluster"].tolist()
print(f"\u2705 Top {N_GAP_CLUSTERS} gap clusters identified: {gap_clusters}")
print()

for cid in gap_clusters:
    stats   = cluster_stats[cluster_stats["cluster"] == cid].iloc[0]
    samples = (
        df[df["cluster"] == cid]["query_text"]
        .sample(
            min(SAMPLE_QUERIES, len(df[df["cluster"] == cid])),
            random_state=RANDOM_SEED
        )
        .tolist()
    )
    print(f"{'=' * 60}")
    print(
        f"  CLUSTER {cid} — Mean Score: {stats['mean_score']:.4f} | "
        f"Miss Rate: {stats['miss_rate']}% | N={int(stats['total_count'])}"
    )
    print(f"{'=' * 60}")
    for i, q in enumerate(samples, 1):
        print(f"  [{i:2d}] {q}")
    print()

# Validation assertion
n_gap_below_thresh = (cluster_stats["mean_score"] < GAP_CLUSTER_THRESH).sum()
assert n_gap_below_thresh >= 2, (
    f"\u274c Expected at least 2 clusters with mean score < {GAP_CLUSTER_THRESH}, "
    f"found {n_gap_below_thresh}.\n"
    f"   Check that retrieval_score column is not already normalised to 1.0."
)
print(
    f"\u2705 Gap cluster assertion passed: "
    f"{n_gap_below_thresh} clusters below {GAP_CLUSTER_THRESH}."
)

---
## 💡 Step 6 — Propose Improvements

### Cell 10 — Diagnose Each Gap Cluster and Write Fixes

### Why this step exists

Identifying that retrieval is failing is only half the work. The value of this analysis is the actionable diagnosis: **why** is retrieval failing for this cluster, and **what specific change** will fix it? Read the 10 sample queries from each gap cluster in Cell 9, infer the topic, and fill in the diagnosis template below.

> ✏️ **Your task:** Fill in `topic_label`, `failure_reason`, and `proposed_fix` for each of the 3 gap clusters identified in Cell 9.

In [ ]:
# ============================================================
# CELL 10 — Gap Cluster Diagnosis
# ✏️  Fill in the three diagnosis blocks below.
# ============================================================

GAP_DIAGNOSES = [
    {
        "cluster_id"    : gap_clusters[0],   # auto-filled from Cell 9
        "topic_label"   : "",                # ✏️ e.g. "Finance Reports"
        "failure_reason": "",                # ✏️ e.g. "No financial documents in corpus"
        "proposed_fix"  : "",                # ✏️ e.g. "Ingest Q1–Q4 financial reports"
    },
    {
        "cluster_id"    : gap_clusters[1],
        "topic_label"   : "",                # ✏️
        "failure_reason": "",                # ✏️
        "proposed_fix"  : "",                # ✏️
    },
    {
        "cluster_id"    : gap_clusters[2],
        "topic_label"   : "",                # ✏️
        "failure_reason": "",                # ✏️
        "proposed_fix"  : "",                # ✏️
    },
]

# --- Validate all fields are filled ------------------------------
REQUIRED_FIELDS = ["topic_label", "failure_reason", "proposed_fix"]
all_filled = True

for i, d in enumerate(GAP_DIAGNOSES):
    for field in REQUIRED_FIELDS:
        if not d.get(field, "").strip():
            print(f"\u26a0\ufe0f  GAP_DIAGNOSES[{i}]['{field}'] is empty — fill it in.")
            all_filled = False

if all_filled:
    print("\u2705 All diagnosis fields completed. Summary:\n")
    for d in GAP_DIAGNOSES:
        stats = cluster_stats[cluster_stats["cluster"] == d["cluster_id"]].iloc[0]
        print(f"  {'=' * 58}")
        print(f"  Cluster {d['cluster_id']} — {d['topic_label']}")
        print(
            f"  Mean score   : {stats['mean_score']:.4f} | "
            f"Miss rate: {stats['miss_rate']}%"
        )
        print(f"  Why failing  : {d['failure_reason']}")
        print(f"  Proposed fix : {d['proposed_fix']}")
        print()
else:
    print()
    print("Fill in the empty fields above and re-run this cell before running Cell 11.")

assert all_filled, (
    "\u274c Diagnosis incomplete. "
    "Fill in all topic_label, failure_reason, and proposed_fix fields."
)

---
## ✅ Cell 11 — Final Validation

All 6 checks must pass before moving to Lab 3B.

| Check | What It Validates |
|---|---|
| `embedding_count` | 500 embeddings produced with no errors |
| `embedding_dim` | Each embedding is 384-dimensional |
| `kmeans_converged` | K-means converged within max_iter |
| `umap_distinct` | UMAP centroids show sufficient spread (≥ 4 distinct clusters) |
| `gap_clusters_found` | At least 2 clusters have mean score < 0.55 |
| `diagnoses_complete` | All 3 gap cluster diagnoses are written |

In [ ]:
# ============================================================
# CELL 11 — Final Validation Runner
# ============================================================
results = {}

# Check 1: embedding count
try:
    results["embedding_count"] = len(embeddings) == len(df)
except Exception:
    results["embedding_count"] = False

# Check 2: embedding dimension
try:
    results["embedding_dim"] = embeddings.shape[1] == EMBED_DIM
except Exception:
    results["embedding_dim"] = False

# Check 3: KMeans converged
try:
    results["kmeans_converged"] = kmeans.n_iter_ < KMEANS_ITER
except Exception:
    results["kmeans_converged"] = False

# Check 4: UMAP distinct clusters
try:
    centroids = np.array([
        [
            df.loc[df["cluster"] == c, "umap_x"].mean(),
            df.loc[df["cluster"] == c, "umap_y"].mean()
        ]
        for c in range(N_CLUSTERS)
    ])
    results["umap_distinct"] = np.std(centroids) > 1.5
except Exception:
    results["umap_distinct"] = False

# Check 5: at least 2 gap clusters below threshold
try:
    n_below = (cluster_stats["mean_score"] < GAP_CLUSTER_THRESH).sum()
    results["gap_clusters_found"] = n_below >= 2
except Exception:
    results["gap_clusters_found"] = False

# Check 6: all diagnoses complete
try:
    results["diagnoses_complete"] = all(
        all(d.get(f, "").strip() for f in REQUIRED_FIELDS)
        for d in GAP_DIAGNOSES
    )
except Exception:
    results["diagnoses_complete"] = False

# --- Print results -----------------------------------------------
print("=" * 50)
print("LAB 3A \u2014 VALIDATION RESULTS (CSV Version)")
print("=" * 50)
all_pass = True
for check, passed in results.items():
    status = "\u2705 PASS" if passed else "\u274c FAIL"
    print(f"  {status}  {check}")
    if not passed:
        all_pass = False
print("=" * 50)
if all_pass:
    print("\U0001f389 ALL CHECKS PASSED \u2014 Ready for Lab 3B!")
else:
    print("\u26a0\ufe0f  Some checks failed \u2014 see FAIL Indicators below.")

---
## ❌ FAIL Indicators & Fixes

| Check | Symptom | Cause | Fix |
|---|---|---|---|
| `embedding_count` | `len(embeddings) != 500` | Rows dropped during cleaning | Check `dropna` in Cell 2; inspect raw CSV |
| `embedding_dim` | Dimension ≠ 384 | Wrong model loaded | Confirm `EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"` |
| `kmeans_converged` | `n_iter_ == max_iter` | K-means did not converge | Increase `KMEANS_ITER` to 1000 or reduce `N_CLUSTERS` to 5 |
| `umap_distinct` | All points in one blob | Embeddings too similar | Increase `UMAP_NEIGHBORS` to 30; check embeddings are not all zeros |
| `gap_clusters_found` | All scores ≥ 0.55 | Scores pre-normalised to 1.0 | Check raw CSV: `df['retrieval_score'].describe()` |
| `diagnoses_complete` | Empty fields | Not filled in | Edit Cell 10 and fill all three diagnosis blocks |

---
## 🏁 Key Takeaways

| Concept | What You Learned | Production Application |
|---|---|---|
| **Score distribution** | Bimodal distribution reveals systematic content gaps | Monitor score distribution daily; alert on miss rate > 20% |
| **Semantic clustering** | Sentence embeddings group queries by intent, not keywords | Use clusters to prioritise corpus expansion |
| **Elbow method** | Validates cluster count before committing to k | Always validate k before production clustering |
| **UMAP overlay** | Spatial + score overlay reveals gap cluster locations | Build interactive dashboards with Plotly for ops teams |
| **Gap diagnosis** | Written diagnosis forces precise root cause identification | Feed diagnoses into corpus management backlog |

---
*Next: Lab 3B — Query Rewriting and Hybrid Search → `03_lab3b_advanced_retrieval.ipynb`*